# Notebook 1 — Multi-UE Traffic Generation

Loads projected synthetic burst pools, learns user-class parameters,
assigns flows to UEs, and applies temporal (request-response) correlation.

```
synth_test_bursts_clean.parquet   (from audit notebook)
        ↓
  ★ THIS NOTEBOOK ★
        ↓
traffic_profiles/run_<apps>_<ts>/bursts/
        ↓
Notebook 2 — MGEN script generation
```

**Prerequisites:** run the audit notebook first so that
`synth_test_bursts_clean.parquet` exists for every app you intend to use.

## Cell 1 — Setup & App Discovery

In [1]:
import json
import math
import warnings
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ── helpers ───────────────────────────────────────────────────────────────────

def find_project_root(start: Optional[Path] = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "data").is_dir() and (cand / "artifacts").is_dir():
            return cand
    raise FileNotFoundError(
        "Could not find project root (needs ./data and ./artifacts)"
    )


def find_latest_run_dir(base_dir: Path) -> Optional[Path]:
    if not base_dir.exists():
        return None
    run_dirs = sorted(
        p for p in base_dir.iterdir()
        if p.is_dir() and p.name.startswith("run_")
    )
    return run_dirs[-1] if run_dirs else None


def discover_apps(artifacts_root: Path) -> Dict[str, Dict]:
    """
    Scan artifacts/ for apps that have both DL and UL burst files.

    File preference order:
      1. synth_test_bursts_mgen.parquet    ← projected (Notebook 0 output)
      2. synth_test_bursts_markov.parquet  ← raw fallback
    """
    priority = [
        "synth_test_bursts_mgen.parquet",
        "synth_test_bursts_markov.parquet",
    ]

    apps: Dict[str, Dict] = {}

    for app_dir in sorted(artifacts_root.iterdir()):
        if not app_dir.is_dir() or app_dir.name.startswith("."):
            continue

        app_name = app_dir.name
        dl_run   = find_latest_run_dir(app_dir / "downlink")
        ul_run   = find_latest_run_dir(app_dir / "uplink")

        dl_file = ul_file = None

        for fname in priority:
            if dl_run and (dl_run / "data" / fname).exists():
                dl_file = dl_run / "data" / fname
                break
        for fname in priority:
            if ul_run and (ul_run / "data" / fname).exists():
                ul_file = ul_run / "data" / fname
                break

        if dl_file:
            apps[app_name] = {
                "dl":      dl_file,
                "ul":      ul_file,
                "app_dir": app_dir,
                "dl_projected": dl_file.name == "synth_test_bursts_mgen.parquet",
                "ul_projected": ul_file is not None and ul_file.name == "synth_test_bursts_mgen.parquet",
            }

    return apps


# ── discover ──────────────────────────────────────────────────────────────────
PROJECT_ROOT  = find_project_root()
ARTIFACTS_ROOT = PROJECT_ROOT / "artifacts"
OUTPUT_ROOT   = PROJECT_ROOT / "traffic_profiles"
OUTPUT_ROOT.mkdir(exist_ok=True)

available_apps = discover_apps(ARTIFACTS_ROOT)

print("="*72)
print("  MULTI-UE TRAFFIC GENERATOR — SETUP")
print("="*72)
print(f"\n  Project root : {PROJECT_ROOT}")
print(f"  Output root  : {OUTPUT_ROOT}")

if not available_apps:
    raise FileNotFoundError(
        "No apps found. Run the audit notebook first to generate "
        "synth_test_bursts_mgen.parquet."
    )

print(f"\n  Found {len(available_apps)} app(s):\n")
for name, meta in sorted(available_apps.items()):
    dl_tag = "✅ projected" if meta["dl_projected"] else "⚠️  raw"
    ul_tag = "✅ projected" if meta["ul_projected"] else "⚠️  raw"
    print(f"  📱 {name.upper()}")
    print(f"     DL : {meta['dl'].name}  {dl_tag}")
    if meta["ul"]:
        print(f"     UL : {meta['ul'].name}  {ul_tag}")
    else:
        print(f"     UL : ⚠️  not found")
    print()

print("="*72)
print("  ✅  Cell 1 complete — ready for configuration")
print("="*72)

  MULTI-UE TRAFFIC GENERATOR — SETUP

  Project root : /Users/ghinwaismail/Desktop/mgen/multimodal-traffic-digital-twins
  Output root  : /Users/ghinwaismail/Desktop/mgen/multimodal-traffic-digital-twins/traffic_profiles

  Found 5 app(s):

  📱 APARAT
     DL : synth_test_bursts_mgen.parquet  ✅ projected
     UL : synth_test_bursts_mgen.parquet  ✅ projected

  📱 FILIMO
     DL : synth_test_bursts_mgen.parquet  ✅ projected
     UL : synth_test_bursts_mgen.parquet  ✅ projected

  📱 IGAP
     DL : synth_test_bursts_mgen.parquet  ✅ projected
     UL : synth_test_bursts_mgen.parquet  ✅ projected

  📱 TELEGRAM
     DL : synth_test_bursts_mgen.parquet  ✅ projected
     UL : synth_test_bursts_mgen.parquet  ✅ projected

  📱 YOUTUBE
     DL : synth_test_bursts_mgen.parquet  ✅ projected
     UL : synth_test_bursts_mgen.parquet  ✅ projected

  ✅  Cell 1 complete — ready for configuration


## Cell 2 — Configuration

All parameters in one place. Copy the recommended values from Cell 8 of
the audit notebook.

In [2]:
import yaml
from pathlib import Path

# ══════════════════════════════════════════════════════════════════════════════
#  LOAD SCENARIO CONFIG
#  All tunable parameters live in scenario_config.yaml next to this notebook.
#  Edit that file, then re-run this cell — no Python edits needed.
# ══════════════════════════════════════════════════════════════════════════════

_cfg_path = PROJECT_ROOT / "scenario_config.yaml"
if not _cfg_path.exists():
    raise FileNotFoundError(
        f"scenario_config.yaml not found at {_cfg_path}\n"
        "Copy it from the repo root next to this notebook."
    )

with open(_cfg_path) as _f:
    _cfg = yaml.safe_load(_f)

# ── Simulation ────────────────────────────────────────────────────────────────
N_UE                = int(_cfg["simulation"]["n_ue"])
SIMULATION_DURATION = float(_cfg["simulation"]["duration"])
RANDOM_SEED         = int(_cfg["simulation"]["random_seed"])
np.random.seed(RANDOM_SEED)

# ── Applications ──────────────────────────────────────────────────────────────
APPS = list(_cfg["apps"])

# ── User classes ──────────────────────────────────────────────────────────────
USER_CLASS_MODE         = "distribution"
USER_CLASS_DISTRIBUTION = {
    k: int(v) for k, v in _cfg["user_classes"]["distribution"].items()
}

USER_CLASS_APP_AFFINITY: Dict[str, Dict] = {
    cls: {"vol_exp": float(vals["vol_exp"]), "int_exp": float(vals["int_exp"])}
    for cls, vals in _cfg["user_classes"]["app_affinity"].items()
}

_raw_override = _cfg["user_classes"].get("app_mix_override") or {}
APP_MIX_OVERRIDE: Dict[str, Dict[str, float]] = {
    cls: {app: float(w) for app, w in apps.items()}
    for cls, apps in _raw_override.items()
} if _raw_override else {}

USER_CLASS_OVERRIDES: Dict = {}   # advanced: leave empty unless needed

# ── Flow sampling ─────────────────────────────────────────────────────────────
SAMPLING_STRATEGY = str(_cfg["sampling_strategy"])

# ── Temporal correlation ──────────────────────────────────────────────────────
_tc = _cfg["temporal_correlation"]
TEMPORAL_CORRELATION = {
    "enabled"                  : bool(_tc["enabled"]),
    "type"                     : "request_response",
    "rtt_delay_range"          : tuple(_tc["rtt_delay_range"]),
    "dl_bursts_per_ul_request" : tuple(_tc["dl_bursts_per_ul_request"]),
    "jitter"                   : float(_tc["jitter"]),
    "mode_overrides"           : dict(_tc.get("mode_overrides") or {}),
    "mode_thresholds"          : dict(_tc.get("mode_thresholds") or {}),
}

# ── Network ───────────────────────────────────────────────────────────────────
_net        = _cfg["network"]
DN_IP       = str(_net["dn_ip"])
UE_IP_PREFIX= str(_net["ue_ip_prefix"])
UE_IP_START = int(_net["ue_ip_start"])
DL_PORT     = int(_net["dl_port"])
UL_PORT     = int(_net["ul_port"])
FLOW_ID_OFFSET = 1000   # internal constant — not exposed in config

# ── Known app modes (curated table — stays in code, not in config) ────────────
# To override one app for this run, add it to mode_overrides in the YAML.
KNOWN_APP_MODES: Dict[str, str] = {
    # Video streaming
    "youtube": "streaming", "netflix": "streaming", "filimo": "streaming",
    "aparat": "streaming", "vimeo": "streaming", "twitch": "streaming",
    "disneyplus": "streaming", "hulu": "streaming", "amazonprime": "streaming",
    "appletv": "streaming", "dailymotion": "streaming",
    # Messaging & social
    "telegram": "request_response", "igap": "request_response",
    "whatsapp": "request_response", "instagram": "request_response",
    "twitter": "request_response", "facebook": "request_response",
    "signal": "request_response", "wechat": "request_response",
    "line": "request_response", "viber": "request_response",
    # Web browsing
    "chrome": "request_response", "firefox": "request_response",
    "safari": "request_response",
    # VoIP & calls
    "zoom": "bidirectional_continuous", "teams": "bidirectional_continuous",
    "skype": "bidirectional_continuous", "meet": "bidirectional_continuous",
    "webex": "bidirectional_continuous", "discord": "bidirectional_continuous",
    # Cloud / upload
    "dropbox": "uplink_push", "googledrive": "uplink_push",
    "onedrive": "uplink_push", "icloud": "uplink_push", "gdrive": "uplink_push",
    # Maps & IoT
    "googlemaps": "iot_periodic", "waze": "iot_periodic",
    "applemaps": "iot_periodic", "mqtt": "iot_periodic", "coap": "iot_periodic",
    # Push notifications
    "fcm": "server_push", "apns": "server_push", "onesignal": "server_push",
    # Gaming
    "fortnite": "bidirectional_continuous", "roblox": "bidirectional_continuous",
    "pubg": "bidirectional_continuous", "clashofclans": "server_push",
}

# ── Validate ──────────────────────────────────────────────────────────────────
for app in APPS:
    if app not in available_apps:
        raise ValueError(
            f"App '{app}' not found in artifacts/. "
            f"Available: {list(available_apps.keys())}"
        )

total_classes = sum(USER_CLASS_DISTRIBUTION.values())
if total_classes != N_UE:
    raise ValueError(
        f"user_classes.distribution sums to {total_classes} "
        f"but simulation.n_ue = {N_UE}. They must match."
    )

print("="*72)
print("  CONFIGURATION  (loaded from scenario_config.yaml)")
print("="*72)
print(f"  Config file  : {_cfg_path.relative_to(PROJECT_ROOT)}")
print(f"  Simulation   : {SIMULATION_DURATION}s  ({SIMULATION_DURATION/60:.1f} min)  |  {N_UE} UEs")
print(f"  Apps         : {', '.join(APPS)}")
print(f"  User classes : {USER_CLASS_DISTRIBUTION}")
print(f"  Sampling     : {SAMPLING_STRATEGY}")
print(f"  Correlation  : "
      f"{'enabled' if TEMPORAL_CORRELATION['enabled'] else 'disabled'}  "
      f"rtt={TEMPORAL_CORRELATION['rtt_delay_range']}")
print(f"  Network      : DN={DN_IP}  "
      f"UEs={UE_IP_PREFIX}{UE_IP_START}–{UE_IP_PREFIX}{UE_IP_START+N_UE-1}")
print(f"  Ports        : DL={DL_PORT}  UL={UL_PORT}")
print(f"  Random seed  : {RANDOM_SEED}")
print("\n  ✅  Cell 2 complete")


FileNotFoundError: scenario_config.yaml not found at /Users/ghinwaismail/Desktop/mgen/multimodal-traffic-digital-twins/scenario_config.yaml
Copy it from the repo root next to this notebook.

## Cell 3 — Load & Analyse Burst Pools

In [ ]:
app_data: Dict[str, Dict] = {}

print("="*72)
print("  LOADING BURST POOLS")
print("="*72)

for app in APPS:
    meta = available_apps[app]

    dl_df = pd.read_parquet(meta["dl"])
    ul_df = pd.read_parquet(meta["ul"]) if meta["ul"] else pd.DataFrame()

    # ── confirm projected columns are present ──────────────────────────────
    
    has_projected_dl = "packets_export" in dl_df.columns
    has_projected_ul = "packets_export" in ul_df.columns if len(ul_df) else False

    app_data[app] = {"dl": dl_df, "ul": ul_df}

    # volume stats
    
    dl_mb = dl_df["bytes"].sum() / 1e6
    ul_mb = ul_df["bytes"].sum() / 1e6 if len(ul_df) else 0
    byte_ratio = dl_mb / ul_mb if ul_mb > 0 else float("inf")

    dl_flows = dl_df["synthetic_flow_id"].nunique()
    ul_flows = ul_df["synthetic_flow_id"].nunique() if len(ul_df) else 0
    flow_ratio = dl_flows / ul_flows if ul_flows > 0 else float("inf")

    print(f"\n  📱 {app.upper()}")
    print(f"     DL : {len(dl_df):>6,} bursts  {dl_flows:>3} flows  "
          f"{dl_mb:>8.2f} MB  "
          f"{'(projected)' if has_projected_dl else '(raw ⚠️)'}")
    print(f"     UL : {len(ul_df):>6,} bursts  {ul_flows:>3} flows  "
          f"{ul_mb:>8.2f} MB  "
          f"{'(projected)' if has_projected_ul else '(raw ⚠️)' if len(ul_df) else '—'}")
    print(f"     DL/UL byte ratio  : {byte_ratio:.1f}:1")
    print(f"     DL/UL flow ratio  : {flow_ratio:.2f}:1")

    if not has_projected_dl:
        print(f"     ⚠️  DL is using raw data — run the audit notebook first.")

    if len(ul_df) == 0:
        print(f"     ⚠️  No UL data — UL traffic will not be generated for {app}.")

print("\n  ✅  Cell 3 complete")

## Cell 4 — Auto-Learn User Class Parameters

Derives realistic heavy / medium / light parameters directly from the
data distributions — flow count ranges, DL/UL weights, cluster sampling
weights. Manual overrides from Cell 2 are applied after.

In [ ]:
def auto_learn_user_classes(
    app_name: str,
    dl_df: pd.DataFrame,
    ul_df: pd.DataFrame,
) -> Dict[str, Dict]:
    """
    Learn heavy / medium / light class parameters from data percentiles
    and cluster distributions.
    """
    # per-flow byte totals for DL
    
    dl_flow = (
        dl_df.groupby("synthetic_flow_id")["bytes"]
             .sum()
    )
    p25 = dl_flow.quantile(0.25)
    p75 = dl_flow.quantile(0.75)

    # DL/UL weight from natural flow ratio
    
    n_dl = dl_df["synthetic_flow_id"].nunique()
    n_ul = ul_df["synthetic_flow_id"].nunique() if len(ul_df) else 1
    flow_ratio = n_dl / n_ul
    dl_w = round(flow_ratio / (flow_ratio + 1), 2)
    ul_w = round(1 - dl_w, 2)

    # cluster weights for DL
    
    cw_dl: Dict[int, float] = {}
    if "cluster" in dl_df.columns:
        avg_bytes = dl_df.groupby("cluster")["bytes"].mean()
        for c, b in avg_bytes.items():
            if b > p75:
                cw_dl[int(c)] = 2.0
            elif b < p25:
                cw_dl[int(c)] = 0.5
            else:
                cw_dl[int(c)] = 1.0

    # cluster weights for UL (uniform — UL is lighter and more balanced)
    
    cw_ul: Dict[int, float] = {}
    if "cluster" in ul_df.columns and len(ul_df) > 0:
        cw_ul = {int(c): 1.0 for c in ul_df["cluster"].unique()}

    classes = {
        "heavy": {
            "total_flows"         : (10, 15),
            "dl_weight"           : dl_w,
            "ul_weight"           : ul_w,
            "apps"                : {app_name: 1.0},
            "sampling_weights_dl" : {k: v * 2.0 if v == 2.0 else v
                                     for k, v in cw_dl.items()},
            "sampling_weights_ul" : cw_ul.copy(),
            "description"         : "Heavy — favours large flows, many connections",
        },
        "medium": {
            "total_flows"         : (6, 10),
            "dl_weight"           : dl_w,
            "ul_weight"           : ul_w,
            "apps"                : {app_name: 1.0},
            "sampling_weights_dl" : {k: 1.0 for k in cw_dl},
            "sampling_weights_ul" : {k: 1.0 for k in cw_ul},
            "description"         : "Medium — balanced sampling",
        },
        "light": {
            "total_flows"         : (2, 5),
            "dl_weight"           : dl_w,
            "ul_weight"           : ul_w,
            "apps"                : {app_name: 1.0},
            "sampling_weights_dl" : {k: 2.0 if v == 0.5 else 0.5
                                     for k, v in cw_dl.items()},
            "sampling_weights_ul" : {k: 1.0 for k in cw_ul},
            "description"         : "Light — few flows, small volumes",
        },
    }
    return classes


learned_classes: Dict[str, Dict] = {}

print("="*72)
print("  AUTO-LEARNING USER CLASS PARAMETERS")
print("="*72)

for app in APPS:
    cl = auto_learn_user_classes(
        app, app_data[app]["dl"], app_data[app]["ul"]
    )

    # apply manual overrides
    
    for cls_name, overrides in USER_CLASS_OVERRIDES.items():
        if cls_name in cl:
            cl[cls_name].update(overrides)

    learned_classes[app] = cl

    print(f"\n  📱 {app.upper()}")
    for cls_name in ["heavy", "medium", "light"]:
        p = cl[cls_name]
        print(f"\n    👤 {cls_name.upper()}")
        print(f"       flows range   : {p['total_flows']}")
        print(f"       DL/UL weights : {p['dl_weight']} / {p['ul_weight']}")
        print(f"       DL clusters   : {p['sampling_weights_dl']}")
        print(f"       note          : {p['description']}")

print("\n  ✅  Cell 4 complete")

## Cell 5 — Assign User Classes & Sample Flows

Assigns user classes to UEs, then samples flows from each app pool using
stratified sampling that preserves the cluster distribution.

==========================================
Flow Sampling with Adaptive Per-Class App Allocation

──────────────────────────────

A — volume_score now uses MEAN bytes per flow, not median.
    Mean correctly reflects the average
    contribution of one flow to total traffic volume.

B — scores are normalised to [0, 1] before applying exponents.
  Without normalisation, the raw score magnitudes (bytes vs tiny
  ratios) interact unpredictably with the exponents. Normalised
  scores make vol_exp and int_exp intuitive: 1.0 = linear preference,
  2.0 = quadratic, -1.0 = strong avoidance.

C — per-app flow count scaled from each app's own flows_range.
    each app draws from its own flows_range (as in the original), 
    then the result is multiplied by (weight × N_apps). At uniform 
    weight this equals 1.0 (no change); at higher weight the app gets
    more flows proportionally.

D — min_weight_floor (default 0.10).
  Prevents any app from being allocated < 10% of what it would get
  under uniform sampling, ensuring every app is meaningfully present.

E — softer default affinity exponents.
  Old defaults (vol_exp=2.0, int_exp=-1.0 for heavy) caused extreme
  concentration. New defaults use 1.5 / -0.5 for heavy, giving a
  clear preference without near-zero allocation for minor apps.

USER_CLASS_APP_AFFINITY in Cell 2 overrides the defaults below.


In [ ]:
import math
import numpy as np
import pandas as pd
from typing import Dict, List, Tuple


# ══════════════════════════════════════════════════════════════════════════════
#  DEFAULT AFFINITY PROFILES  (override in Cell 2 via USER_CLASS_APP_AFFINITY)
# ══════════════════════════════════════════════════════════════════════════════
#
#  vol_exp > 0  prefer high-volume apps (streaming)
#  vol_exp < 0  prefer low-volume apps  (messaging, IoT)
#  int_exp > 0  prefer interactive apps (messaging)
#  int_exp < 0  prefer bulk apps        (streaming)
#
#  Softer defaults than v1 to avoid near-zero allocations:
#
_DEFAULT_APP_AFFINITY: Dict[str, Dict] = {
    "heavy" : {"vol_exp":  1.5, "int_exp": -0.5},  # streaming-leaning
    "medium": {"vol_exp":  1.0, "int_exp":  0.0},  # neutral
    "light" : {"vol_exp": -0.3, "int_exp":  0.5},  # messaging-leaning
}

# Minimum fraction of uniform allocation any app can receive.
# 0.10 means no app gets less than 10% of what uniform sampling would give.
_MIN_WEIGHT_FLOOR = 0.10


# ══════════════════════════════════════════════════════════════════════════════
#  PART 1 — APP ALLOCATION SCORES
# ══════════════════════════════════════════════════════════════════════════════

def compute_app_allocation_scores(dl_df: pd.DataFrame) -> Dict:
    """
    Compute two normalisation-ready scores for one app from its DL pool.
    Called once per app; reused for all UEs.

    volume_score
        MEAN total bytes produced by a single DL flow.
        Mean correctly reflects average flow contribution
        even when the distribution is skewed.
        High  → bulk / streaming app.
        Low   → lightweight / control app.

    interactivity_score
        Mean bursts per flow divided by mean bytes per burst.
        High  → many small events per flow (messaging, IoT).
        Low   → few large bursts per flow  (streaming, backup).

    Both are returned as raw values; normalisation happens in
    compute_app_weights so all apps are on the same scale.
    """
    if dl_df.empty:
        return {"volume_score": 1.0, "interactivity_score": 1.0}

    # mean bytes per flow
    bytes_per_flow = float(
        dl_df.groupby("synthetic_flow_id")["bytes"].sum().mean()
    )

    # mean bursts per flow
    bursts_per_flow = float(
        dl_df.groupby("synthetic_flow_id").size().mean()
    )

    # mean bytes per burst
    bytes_per_burst = float(dl_df["bytes"].mean())

    interactivity = (
        bursts_per_flow / bytes_per_burst
        if bytes_per_burst > 0 else 1.0
    )

    return {
        "volume_score"       : max(bytes_per_flow,  1.0),
        "interactivity_score": max(interactivity,   1e-9),
    }


def compute_app_weights(
    apps: List[str],
    app_scores: Dict[str, Dict],
    cls_name: str,
    affinity_config: Dict,
    min_floor: float = _MIN_WEIGHT_FLOOR,
) -> Dict[str, float]:
    """
    Derive normalised per-app allocation weights for one user class.

    Manual override (APP_MIX_OVERRIDE in Cell 2) takes priority when set
    for this class.  Any app not listed in the override gets the floor
    weight so it remains present at a minimal level.

    Adaptive path (default when no override):
      1. Normalise each score to [0, 1] across apps.
      2. raw_weight = norm_vol ^ vol_exp × norm_int ^ int_exp
      3. Apply min_floor: no weight below (min_floor / N_apps).
      4. Re-normalise to sum to 1.
    """
    n = len(apps)

    # ── manual override takes priority ────────────────────────────────────
    # APP_MIX_OVERRIDE is defined in Cell 2.  Values are relative weights
    # (percentages, fractions — anything); they are normalised here.
    # Apps not listed in the override receive the floor weight.

    
    override = APP_MIX_OVERRIDE.get(cls_name)
    if override:
        floor_val  = min_floor / n
        
        # scale floor to be relative to the override magnitudes so it is
        # meaningful regardless of whether the user typed 0–1 or 0–100
        
        ref_scale  = sum(override.values()) / len(override) if override else 1.0
        raw: Dict[str, float] = {
            app: float(override.get(app, floor_val * ref_scale))
            for app in apps
        }
        total = sum(raw.values())
        return {app: w / total for app, w in raw.items()}

    # ── adaptive path ─────────────────────────────────────────────────────
    
    aff     = affinity_config.get(cls_name, {"vol_exp": 1.0, "int_exp": 0.0})
    vol_exp = aff["vol_exp"]
    int_exp = aff["int_exp"]

    # normalise scores to [0, 1] across apps
    
    max_vol = max(app_scores[a]["volume_score"]        for a in apps)
    max_int = max(app_scores[a]["interactivity_score"] for a in apps)

    raw: Dict[str, float] = {}
    for app in apps:
        # clamp to avoid 0^negative = inf; floor at a tiny epsilon
        
        norm_vol = max(app_scores[app]["volume_score"]        / max_vol, 1e-6)
        norm_int = max(app_scores[app]["interactivity_score"] / max_int, 1e-6)
        raw[app] = (norm_vol ** vol_exp) * (norm_int ** int_exp)

    # apply minimum floor: no app gets less than min_floor / n of total weight
    
    floor_val = min_floor / n
    raw = {app: max(w, floor_val) for app, w in raw.items()}

    total = sum(raw.values())
    return {app: w / total for app, w in raw.items()}


# ══════════════════════════════════════════════════════════════════════════════
#  PART 2 — STRATIFIED FLOW SAMPLER 
# ══════════════════════════════════════════════════════════════════════════════

def sample_flows_stratified(
    df: pd.DataFrame,
    n_flows: int,
    cluster_weights: Dict[int, float],
) -> List[int]:
    """
    Sample n_flows flow IDs weighted by cluster.
    With replacement so n_flows can exceed pool size.
    """
    if len(df) == 0 or n_flows == 0:
        return []
    available = df["synthetic_flow_id"].unique()
    if not cluster_weights or "cluster" not in df.columns:
        return np.random.choice(
            available, size=n_flows, replace=True
        ).tolist()
    dom_cluster = (
        df.groupby("synthetic_flow_id")["cluster"]
          .agg(lambda x: x.mode().iat[0])
    )
    weights = dom_cluster.map(
        lambda c: cluster_weights.get(int(c), 1.0)
    )
    weights = weights / weights.sum()
    return np.random.choice(
        available, size=n_flows, replace=True, p=weights.values
    ).tolist()


# ══════════════════════════════════════════════════════════════════════════════
#  PART 3 — COMPUTE APP SCORES ONCE FROM FULL POOLS
# ══════════════════════════════════════════════════════════════════════════════

app_scores: Dict[str, Dict] = {
    app: compute_app_allocation_scores(app_data[app]["dl"])
    for app in APPS
}

affinity_config: Dict = globals().get(
    "USER_CLASS_APP_AFFINITY", _DEFAULT_APP_AFFINITY
)
n_apps = len(APPS)

# ── allocation score report ───────────────────────────────────────────────────

print("=" * 72)
print("  APP ALLOCATION SCORES  (data-driven, no app names)")
print("=" * 72)
print(f"\n  {'App':<12}  {'vol_score (mean B/flow)':>24}  "
      f"{'interactivity':>15}")
print(f"  {'-'*12}  {'-'*24}  {'-'*15}")
for app in APPS:
    s = app_scores[app]
    print(f"  {app:<12}  {s['volume_score']:>24,.0f}  "
          f"{s['interactivity_score']:>15.6f}")

print(f"\n  Affinity profiles (vol_exp, int_exp per class):")
for cls_name in USER_CLASS_DISTRIBUTION:
    aff = affinity_config.get(cls_name, {"vol_exp": 1.0, "int_exp": 0.0})
    print(f"    {cls_name:<10}  vol_exp={aff['vol_exp']:+.1f}  "
          f"int_exp={aff['int_exp']:+.1f}")

print(f"\n  Derived app weights per class "
      f"(min floor={_MIN_WEIGHT_FLOOR}, uniform=1/{n_apps}={1/n_apps:.3f}):")
for cls_name in USER_CLASS_DISTRIBUTION:
    w = compute_app_weights(
        APPS, app_scores, cls_name, affinity_config, _MIN_WEIGHT_FLOOR
    )
    print(f"  {cls_name}:")
    for app, wt in w.items():
        bar  = "█" * int(wt * 40)
        flag = " ← floor" if wt <= _MIN_WEIGHT_FLOOR / n_apps * 1.01 else ""
        print(f"    {app:<12}  {wt:.3f}  {bar}{flag}")

# ══════════════════════════════════════════════════════════════════════════════
#  PART 4 — ASSIGN CLASSES TO UEs
# ══════════════════════════════════════════════════════════════════════════════

ue_classes: List[str] = []
for cls_name, count in USER_CLASS_DISTRIBUTION.items():
    ue_classes.extend([cls_name] * count)
np.random.shuffle(ue_classes)

print("\n" + "=" * 72)
print("  UE CLASS ASSIGNMENTS")
print("=" * 72)
for i, cls in enumerate(ue_classes):
    print(f"  ue{i+1} : {cls}")


# ══════════════════════════════════════════════════════════════════════════════
#  PART 5 — FLOW SAMPLING WITH ADAPTIVE APP WEIGHTS
# ══════════════════════════════════════════════════════════════════════════════
#
#  For each app the base flow count n_base is drawn from that app's own
#  learned flows_range (preserving the original scale).  It is then
#  multiplied by (weight × N_apps):
#
#    n_app = round(n_base × weight × N_apps)
#
#  At uniform weight (1/N_apps), weight × N_apps = 1.0 → no change.
#  At weight=0.4 with N_apps=5, the multiplier is 2.0 → double flows.
#  At weight=0.05 (floor), the multiplier is 0.25 → quarter flows, ≥ 1.
#
# ═════════════════════════════════════════════════════════════════════════════

ue_flow_assignments: Dict[str, Dict] = {}

print("\n" + "=" * 72)
print("  FLOW SAMPLING")
print("=" * 72)

for i, cls_name in enumerate(ue_classes):
    ue_name = f"ue{i+1}"

    app_weights = compute_app_weights(
        APPS, app_scores, cls_name, affinity_config, _MIN_WEIGHT_FLOOR
    )

    flows_by_app: Dict[str, Dict] = {}

    for app in APPS:
        dl_df = app_data[app]["dl"]
        ul_df = app_data[app]["ul"]

        if app not in learned_classes or cls_name not in learned_classes[app]:
            flows_by_app[app] = {"dl_flows": [], "ul_flows": []}
            continue

        p = learned_classes[app][cls_name]

        # base flow count from app's learned range
        lo, hi  = p["total_flows"]
        n_base  = np.random.randint(lo, hi + 1)

        # scale by weight relative to uniform
        scale   = app_weights[app] * n_apps
        n_app   = max(1, round(n_base * scale))

        n_dl = max(1, int(n_app * p["dl_weight"]))
        n_ul = max(0, int(n_app * p["ul_weight"]))
        if len(ul_df) == 0:
            n_ul = 0

        dl_flows = sample_flows_stratified(
            dl_df, n_dl, p.get("sampling_weights_dl", {})
        )
        ul_flows = sample_flows_stratified(
            ul_df, n_ul, p.get("sampling_weights_ul", {})
        )

        flows_by_app[app] = {"dl_flows": dl_flows, "ul_flows": ul_flows}

    ue_flow_assignments[ue_name] = {
        "class" : cls_name,
        "flows" : flows_by_app,
    }

    total_dl = sum(len(f["dl_flows"]) for f in flows_by_app.values())
    total_ul = sum(len(f["ul_flows"]) for f in flows_by_app.values())

    print(f"\n  {ue_name} ({cls_name}):")
    for app, fa in flows_by_app.items():
        wt    = app_weights[app]
        scale = wt * n_apps
        print(f"    {app:<12} : DL {len(fa['dl_flows']):>3} flows  "
              f"UL {len(fa['ul_flows']):>3} flows  "
              f"(w={wt:.3f} → ×{scale:.2f})")
    print(f"    {'total':<12} : DL {total_dl}   UL {total_ul}")

print("\n  ✅  Cell 5 complete")

## Cell 6 — Time-Align Bursts with Temporal Correlation

Extracts bursts for each UE's assigned flows and builds a concrete
timeline using request-response correlation:

```
t=0.000  UL request burst  (small, goes first)
t=0.030  DL response batch starts  (RTT = 30 ms)
t=1.500  DL response batch ends
t=2.200  UL request burst  (next request)
…
```

### Improvement — per-flow start offset

The audit notebook fills `off_dur_s = NaN` with `0.0`, which means each
flow's first burst has no built-in delay. If a heavy flow's total duration
equals or exceeds `SIMULATION_DURATION`, `max_start` becomes 0 and the
old code set `flow_start = 0.0` exactly. Multiple such flows then all fire
at `t=0` simultaneously. The fix adds a small random jitter in that case.



Time-Align Bursts with Auto-Detected Temporal Correlation

──────────────────────────────────────────────────────────────────────

★ streaming classifier no longer relies solely on
   median_dl_on_dur.  Short per-burst durations are normal for
   chunk-based video (each chunk is a small burst); the real signal
   is high byte_ratio + near-MTU packet size.  The new rule is:

     byte_ratio ≥ streaming_byte_ratio
     AND (median_dl_on_dur ≥ streaming_dl_dur_min
          OR  median_pkt_size ≥ streaming_pkt_size_min)

   This correctly captures Filimo/YouTube even when individual bursts
   are short.  streaming_pkt_size_min = 900 B (near-MTU = bulk data).

★ span display shows min_start–max_end instead of 0–max_end.
   A UE with one burst at t=200s was previously shown as "0–200s",
   implying continuous activity.  Now it shows the true active window.

──────────────────────────
1.  Mode detected once per app, reused for all UEs.
2.  server_push uses normalised ul_byte_share. bidirectional_continuous requires both DL and UL cv_gap evidence.
3.  True local contiguous subsequences in request_response.
4.  Real independent mode for correlation-disabled case.
5.  _place_bursts_sequentially tracks prev_end (no intra-flow overlap).
6.  align_server_push: end = start + fixed duration.
7.  streaming checked before server_push in decision tree.
8.  Shared DL/UL scale factor in request_response.
9.  First burst of each request_response batch skips inherited off_dur_s.
10. flow_ratio removed from fingerprint (unused).
11. Manual overrides validated against VALID_MODES.
12. iot_periodic flows staggered across simulation window.

In [ ]:
import math
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd


VALID_MODES = {
    "streaming",
    "request_response",
    "bidirectional_continuous",
    "uplink_push",
    "iot_periodic",
    "server_push",
    "independent",
}


# ══════════════════════════════════════════════════════════════════════════════
#  PART 1 — MODE CLASSIFIER
# ══════════════════════════════════════════════════════════════════════════════

def compute_app_fingerprint(
    dl_df: pd.DataFrame,
    ul_df: pd.DataFrame,
) -> Dict:
    
    """
    Compute statistics from the FULL app pool
    All values are normalised so they are not sensitive to sample size.

    byte_ratio        DL bytes / UL bytes  (inf when UL is empty)
    ul_byte_share     UL bytes / (DL + UL bytes)  — server_push indicator
    median_dl_on_dur  median DL burst active duration (seconds)
    median_pkt_size   median implied packet size across DL bursts (bytes)
                      ★ now used as primary streaming indicator alongside
                        byte_ratio, replacing sole reliance on burst duration
    dl_cv_gap         coefficient of variation of DL inter-burst gaps
    ul_cv_gap         coefficient of variation of UL inter-burst gaps
    """
    
    dl_bytes = float(dl_df["bytes"].sum()) if not dl_df.empty else 0.0
    ul_bytes = float(ul_df["bytes"].sum()) if not ul_df.empty else 0.0
    total    = dl_bytes + ul_bytes

    byte_ratio    = dl_bytes / ul_bytes if ul_bytes > 0 else float("inf")
    ul_byte_share = ul_bytes / total    if total    > 0 else 0.0

    def _cv_gap(df: pd.DataFrame) -> float:
        if df.empty:
            return 0.0
        gaps = df["off_dur_s"].dropna()
        if len(gaps) > 1 and gaps.mean() > 0:
            return float(gaps.std() / gaps.mean())
        return 0.0

    def _median_on_dur(df: pd.DataFrame) -> float:
        return float(df["on_dur_s"].median()) if not df.empty else 0.0

    def _median_pkt_size(df: pd.DataFrame) -> float:
        if df.empty:
            return 0.0
        if "pkt_size_export" in df.columns:
            return float(df["pkt_size_export"].median())
        raw = df["bytes"] / df["packets"].replace(0, np.nan)
        return float(raw.median())

    return {
        "byte_ratio"       : byte_ratio,
        "ul_byte_share"    : ul_byte_share,
        "median_dl_on_dur" : _median_on_dur(dl_df),
        "median_pkt_size"  : _median_pkt_size(dl_df),
        "dl_cv_gap"        : _cv_gap(dl_df),
        "ul_cv_gap"        : _cv_gap(ul_df),
    }


def detect_mode(fp: Dict, thresholds: Dict) -> Tuple[str, str]:
    
    """
    Map a fingerprint to a traffic mode.
    Returns (mode_name, reason) for logging.

    Decision order (most specific first):
      1. iot_periodic           — regular cadence + small packets
      2. streaming              — checked before server_push (Fix 7)
                                  ★ now triggers on high byte_ratio AND
                                  (long burst OR near-MTU packet size)
                                  This captures chunk-based video (Filimo,
                                  YouTube) even when per-burst duration is
                                  short.
      3. server_push            — UL negligible AND not streaming
      4. uplink_push            — UL dominates
      5. bidirectional_continuous — balanced + both sides regular
      6. request_response       — default
    """
    
    br  = fp["byte_ratio"]
    ubs = fp["ul_byte_share"]
    dur = fp["median_dl_on_dur"]
    pkt = fp["median_pkt_size"]
    dcv = fp["dl_cv_gap"]
    ucv = fp["ul_cv_gap"]
    t   = thresholds

    # 1. iot_periodic
    if dcv <= t["iot_cv_gap_max"] and pkt <= t["iot_pkt_size_max"]:
        return (
            "iot_periodic",
            f"dl_cv_gap={dcv:.2f} ≤ {t['iot_cv_gap_max']} "
            f"AND median_pkt_size={pkt:.0f}B ≤ {t['iot_pkt_size_max']}B",
        )

    #  streaming — checked before server_push
    #  near-MTU packet size is a reliable bulk/video
    # indicator even when individual burst durations are short (video chunks).
    # median_dl_on_dur alone is not sufficient for chunk-based streaming.

    
    streaming_signal = (
        dur >= t["streaming_dl_dur_min"]
        or pkt >= t["streaming_pkt_size_min"]
    )
    if br >= t["streaming_byte_ratio"] and streaming_signal:
        reason_signal = (
            f"median_dl_on_dur={dur:.3f}s ≥ {t['streaming_dl_dur_min']}s"
            if dur >= t["streaming_dl_dur_min"]
            else f"median_pkt_size={pkt:.0f}B ≥ {t['streaming_pkt_size_min']}B"
        )
        return (
            "streaming",
            f"byte_ratio={br:.1f} ≥ {t['streaming_byte_ratio']} "
            f"AND ({reason_signal})",
        )

    # 3. server_push — only reached if not streaming
    
    if ubs <= t["server_push_ul_share_max"]:
        return (
            "server_push",
            f"ul_byte_share={ubs:.3f} ≤ {t['server_push_ul_share_max']} "
            f"(UL negligible, not streaming)",
        )

    # 4. uplink_push
    if br <= t["uplink_push_ratio"]:
        return (
            "uplink_push",
            f"byte_ratio={br:.2f} ≤ {t['uplink_push_ratio']}",
        )

    # 5. bidirectional_continuous — requires both DL and UL evidence
    
    if (br <= 3.0
            and dcv <= t["bidir_cv_gap_max"]
            and ucv <= t["bidir_cv_gap_max"]):
        return (
            "bidirectional_continuous",
            f"byte_ratio={br:.2f} ≤ 3.0 "
            f"AND dl_cv_gap={dcv:.2f} ≤ {t['bidir_cv_gap_max']} "
            f"AND ul_cv_gap={ucv:.2f} ≤ {t['bidir_cv_gap_max']}",
        )

    # 6. request_response — default
    
    return (
        "request_response",
        f"byte_ratio={br:.2f}  dl_cv_gap={dcv:.2f}  ul_cv_gap={ucv:.2f} "
        f"(default — no stronger pattern matched)",
    )


_DEFAULT_THRESHOLDS = {
    "iot_cv_gap_max"          : 0.3,
    "iot_pkt_size_max"        : 300,    # bytes
    "streaming_byte_ratio"    : 10.0,
    "streaming_dl_dur_min"    : 0.05,   # seconds  (long-burst path)
    "streaming_pkt_size_min"  : 900,    # bytes    ★ near-MTU path 
    "uplink_push_ratio"       : 0.3,
    "bidir_cv_gap_max"        : 0.6,
    "server_push_ul_share_max": 0.02,
}


def resolve_app_modes(apps, app_data, params):
    overrides  = params.get("mode_overrides", {})
    thresholds = {**_DEFAULT_THRESHOLDS, **params.get("mode_thresholds", {})}
    known      = globals().get("KNOWN_APP_MODES", {})

    bad = {k: v for k, v in overrides.items() if v not in VALID_MODES}
    if bad:
        raise ValueError(f"Invalid mode(s) in mode_overrides: {bad}")

    result = {}
    for app in apps:
        if app in overrides:
            result[app] = (overrides[app], "manual override")
        elif app in known:
            result[app] = (known[app], f"known app table")
        else:
            fp           = compute_app_fingerprint(
                app_data[app]["dl"], app_data[app]["ul"])
            mode, reason = detect_mode(fp, thresholds)
            result[app]  = (mode, f"auto-detected: {reason}")
    return result

# ══════════════════════════════════════════════════════════════════════════════
#  PART 2 — SHARED HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def extract_bursts(df: pd.DataFrame, flow_ids: List[int]) -> pd.DataFrame:
    if df.empty or not flow_ids:
        return pd.DataFrame()
    return (
        df[df["synthetic_flow_id"].isin(flow_ids)]
          .copy()
          .sort_values(["synthetic_flow_id", "synthetic_burst_idx"])
    )


def _safe_flow_start(
    flow_dur: float,
    simulation_duration: float,
    start_offset: float = 0.0,
) -> float:
    max_start = max(0.0, simulation_duration - flow_dur)
    if max_start > 0:
        return float(np.random.uniform(start_offset, start_offset + max_start))
    jitter_ceil = min(5.0, simulation_duration * 0.05)
    return float(np.random.uniform(0.0, jitter_ceil))


def _place_bursts_sequentially(
    burst_df: pd.DataFrame,
    flow_start: float,
    jitter: float = 0.0,
) -> List[Dict]:
    """
    Walk bursts in order assigning absolute timestamps.
    prev_end lower-bound prevents negative-jitter intra-flow overlap.
    """
    records  = []
    t        = flow_start
    prev_end = flow_start

    for _, burst in burst_df.iterrows():
        t        += float(burst["off_dur_s"])
        abs_start = max(prev_end, t + np.random.uniform(-jitter, jitter))
        abs_end   = abs_start + float(burst["on_dur_s"])
        records.append({
            **burst.to_dict(),
            "absolute_start_time": abs_start,
            "absolute_end_time"  : abs_end,
            "flow_start_time"    : flow_start,
        })
        t        = abs_end
        prev_end = abs_end

    return records


def _scale_to_window(
    df: pd.DataFrame,
    simulation_duration: float,
    scale: float = 0.0,
) -> pd.DataFrame:
    """
    scale=0.0  → compute factor from df (independent scaling).
    scale>0.0  → apply given factor (shared scaling for correlated modes).
    """
    if df.empty:
        return df
    if scale <= 0.0:
        max_t = df["absolute_end_time"].max()
        if max_t <= simulation_duration:
            return df
        scale = simulation_duration * 0.98 / max_t

    df = df.copy()
    for col in ["absolute_start_time", "absolute_end_time", "flow_start_time"]:
        if col in df.columns:
            df[col] *= scale
    return df


def _shared_scale(
    dl: pd.DataFrame,
    ul: pd.DataFrame,
    simulation_duration: float,
) -> float:
    max_t = 0.0
    if not dl.empty:
        max_t = max(max_t, float(dl["absolute_end_time"].max()))
    if not ul.empty:
        max_t = max(max_t, float(ul["absolute_end_time"].max()))
    if max_t <= simulation_duration:
        return 0.0
    return simulation_duration * 0.98 / max_t


def _make_minimal_ul_burst(batch_idx: int, t_start: float = 0.0) -> Dict:
    """Synthetic UL placeholder; end = start + fixed 1 ms."""
    duration = 0.001
    return {
        "synthetic_flow_id"  : 99999,
        "synthetic_burst_idx": batch_idx,
        "on_dur_s"           : duration,
        "off_dur_s"          : 0.0,
        "bytes"              : 100,
        "packets"            : 1,
        "packets_export"     : 1,
        "pkt_size_export"    : 100,
        "pps"                : 1000.0,
        "absolute_start_time": t_start,
        "absolute_end_time"  : t_start + duration,
        "flow_start_time"    : t_start,
    }


# ══════════════════════════════════════════════════════════════════════════════
#  PART 3 — MODE-SPECIFIC ALIGNMENT FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════════

def align_independent(
    dl_bursts: pd.DataFrame,
    ul_bursts: pd.DataFrame,
    simulation_duration: float,
    params: Dict,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    jitter = params.get("jitter", 0.005)
    records_dl, records_ul = [], []

    for burst_df, records in [
        (dl_bursts, records_dl),
        (ul_bursts, records_ul),
    ]:
        if burst_df.empty:
            continue
        for _, flow_df in burst_df.groupby("synthetic_flow_id"):
            flow_df  = flow_df.sort_values("synthetic_burst_idx")
            flow_dur = (flow_df["on_dur_s"].sum()
                        + flow_df["off_dur_s"].fillna(0).sum())
            fs = _safe_flow_start(flow_dur, simulation_duration)
            records.extend(_place_bursts_sequentially(flow_df, fs, jitter))

    return (
        _scale_to_window(
            pd.DataFrame(records_dl) if records_dl else pd.DataFrame(),
            simulation_duration,
        ),
        _scale_to_window(
            pd.DataFrame(records_ul) if records_ul else pd.DataFrame(),
            simulation_duration,
        ),
    )


def align_streaming(
    dl_bursts: pd.DataFrame,
    ul_bursts: pd.DataFrame,
    simulation_duration: float,
    params: Dict,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    DL flow order and inter-burst gaps preserved per flow.
    UL placed per-flow with independent random start offsets — same
    method as DL. This prevents intra-flow overlaps that the old
    global-counter spacing could produce when jitter + on_dur_s caused
    consecutive bursts from the same flow to collide.
    Directions scaled independently (no causal coupling).
    """
    jitter     = params.get("jitter", 0.005)
    records_dl = []

    if not dl_bursts.empty:
        for _, flow_df in dl_bursts.groupby("synthetic_flow_id"):
            flow_df  = flow_df.sort_values("synthetic_burst_idx")
            flow_dur = (flow_df["on_dur_s"].sum()
                        + flow_df["off_dur_s"].fillna(0).sum())
            fs = _safe_flow_start(flow_dur, simulation_duration)
            records_dl.extend(_place_bursts_sequentially(flow_df, fs, jitter))

    records_ul = []
    if not ul_bursts.empty:
        for _, flow_df in ul_bursts.groupby("synthetic_flow_id"):
            flow_df  = flow_df.sort_values("synthetic_burst_idx")
            flow_dur = (flow_df["on_dur_s"].sum()
                        + flow_df["off_dur_s"].fillna(0).sum())
            fs = _safe_flow_start(flow_dur, simulation_duration)
            records_ul.extend(_place_bursts_sequentially(flow_df, fs, jitter))

    return (
        _scale_to_window(
            pd.DataFrame(records_dl) if records_dl else pd.DataFrame(),
            simulation_duration,
        ),
        _scale_to_window(
            pd.DataFrame(records_ul) if records_ul else pd.DataFrame(),
            simulation_duration,
        ),
    )

    

def align_request_response(
    dl_bursts: pd.DataFrame,
    ul_bursts: pd.DataFrame,
    simulation_duration: float,
    params: Dict,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    UL request → RTT delay → DL response batch.
    Per-flow cursor for true contiguous subsequences.
    batch_size sampled per batch.
    First burst of each batch skips inherited off_dur_s.
    Shared DL/UL scale factor.
    """
    if dl_bursts.empty:
        return pd.DataFrame(), pd.DataFrame()

    rtt_min, rtt_max       = params["rtt_delay_range"]
    dl_per_min, dl_per_max = params["dl_bursts_per_ul_request"]
    jitter                 = params.get("jitter", 0.005)

    flow_bursts: Dict[int, List] = {}
    for fid, fdf in dl_bursts.groupby("synthetic_flow_id"):
        flow_bursts[int(fid)] = (
            fdf.sort_values("synthetic_burst_idx")
               .reset_index(drop=True)
               .to_dict("records")
        )

    cursors: Dict[int, int] = {fid: 0 for fid in flow_bursts}

    def _active_flows() -> List[int]:
        return [f for f, c in cursors.items() if c < len(flow_bursts[f])]

    ul_pool    = ul_bursts.reset_index(drop=True) if not ul_bursts.empty else None
    current_t  = 0.0
    corr_dl, corr_ul = [], []
    batch_idx  = 0

    while True:
        active = _active_flows()
        if not active:
            break

        chosen_fid          = int(np.random.choice(active))
        cur                 = cursors[chosen_fid]
        bs                  = np.random.randint(dl_per_min, dl_per_max + 1)
        dl_batch            = flow_bursts[chosen_fid][cur: cur + bs]
        cursors[chosen_fid] = cur + len(dl_batch)

        if not dl_batch:
            continue

        if ul_pool is not None and len(ul_pool) > 0:
            ul_row = ul_pool.iloc[batch_idx % len(ul_pool)].to_dict()
        else:
            ul_row = _make_minimal_ul_burst(batch_idx, current_t)

        ul_start = max(0.0, current_t + np.random.uniform(-jitter, jitter))
        ul_end   = ul_start + float(ul_row["on_dur_s"])
        corr_ul.append({
            **ul_row,
            "absolute_start_time" : ul_start,
            "absolute_end_time"   : ul_end,
            "correlated_batch_id" : batch_idx,
            "timeline_flow_id"    : f"ul_batch_{batch_idx}",
        })

        rtt      = np.random.uniform(rtt_min, rtt_max)
        t        = ul_end + rtt
        prev_end = t

        for burst_idx_in_batch, burst_row in enumerate(dl_batch):
            if burst_idx_in_batch > 0:
                t += float(burst_row["off_dur_s"])

            abs_start = max(prev_end, t + np.random.uniform(-jitter, jitter))
            abs_end   = abs_start + float(burst_row["on_dur_s"])
            corr_dl.append({
                **burst_row,
                "absolute_start_time" : abs_start,
                "absolute_end_time"   : abs_end,
                "correlated_batch_id" : batch_idx,
                "timeline_flow_id"    : f"dl_{chosen_fid}_batch_{batch_idx}",
            })
            t        = abs_end
            prev_end = abs_end

        current_t = t + np.random.uniform(0.1, 2.0)
        
        # No wrap-around reset. The shared scale factor applied after the
        # loop brings everything within the window. Resetting current_t
        # backward would place later batches before earlier ones, causing
        # the same synthetic_flow_id to appear out-of-order and creating
        # false intra-flow overlaps in validation.

        batch_idx += 1

    dl_out = pd.DataFrame(corr_dl) if corr_dl else pd.DataFrame()
    ul_out = pd.DataFrame(corr_ul) if corr_ul else pd.DataFrame()

    scale = _shared_scale(dl_out, ul_out, simulation_duration)
    dl_out = _scale_to_window(dl_out, simulation_duration, scale)
    ul_out = _scale_to_window(ul_out, simulation_duration, scale)

    return dl_out, ul_out


def align_bidirectional_continuous(
    dl_bursts: pd.DataFrame,
    ul_bursts: pd.DataFrame,
    simulation_duration: float,
    params: Dict,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    return align_independent(dl_bursts, ul_bursts, simulation_duration, params)


def align_uplink_push(
    dl_bursts: pd.DataFrame,
    ul_bursts: pd.DataFrame,
    simulation_duration: float,
    params: Dict,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    jitter     = params.get("jitter", 0.005)
    records_ul = []

    if not ul_bursts.empty:
        for _, flow_df in ul_bursts.groupby("synthetic_flow_id"):
            flow_df  = flow_df.sort_values("synthetic_burst_idx")
            flow_dur = (flow_df["on_dur_s"].sum()
                        + flow_df["off_dur_s"].fillna(0).sum())
            fs = _safe_flow_start(flow_dur, simulation_duration)
            records_ul.extend(_place_bursts_sequentially(flow_df, fs, jitter))

    records_dl = []
    if not dl_bursts.empty:
        for _, flow_df in dl_bursts.groupby("synthetic_flow_id"):
            flow_df  = flow_df.sort_values("synthetic_burst_idx")
            flow_dur = (flow_df["on_dur_s"].sum()
                        + flow_df["off_dur_s"].fillna(0).sum())
            fs = _safe_flow_start(flow_dur, simulation_duration)
            records_dl.extend(_place_bursts_sequentially(flow_df, fs, jitter))

    return (
        _scale_to_window(
            pd.DataFrame(records_dl) if records_dl else pd.DataFrame(),
            simulation_duration,
        ),
        _scale_to_window(
            pd.DataFrame(records_ul) if records_ul else pd.DataFrame(),
            simulation_duration,
        ),
    )


def align_iot_periodic(
    dl_bursts: pd.DataFrame,
    ul_bursts: pd.DataFrame,
    simulation_duration: float,
    params: Dict,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Flows staggered across the first 10 % of the window (Fix 12)."""
    jitter   = min(params.get("jitter", 0.005), 0.002)
    records_dl, records_ul = [], []

    for burst_df, records in [
        (dl_bursts, records_dl),
        (ul_bursts, records_ul),
    ]:
        if burst_df.empty:
            continue
        flow_ids       = sorted(burst_df["synthetic_flow_id"].unique())
        n_flows        = len(flow_ids)
        stagger_window = min(simulation_duration * 0.10, 5.0)

        for idx, fid in enumerate(flow_ids):
            flow_df = (
                burst_df[burst_df["synthetic_flow_id"] == fid]
                .sort_values("synthetic_burst_idx")
            )
            base = (idx / max(n_flows, 1)) * stagger_window
            fs   = base + np.random.uniform(
                0.0, stagger_window / max(n_flows, 1)
            )
            records.extend(_place_bursts_sequentially(flow_df, fs, jitter))

    return (
        _scale_to_window(
            pd.DataFrame(records_dl) if records_dl else pd.DataFrame(),
            simulation_duration,
        ),
        _scale_to_window(
            pd.DataFrame(records_ul) if records_ul else pd.DataFrame(),
            simulation_duration,
        ),
    )


def align_server_push(
    dl_bursts: pd.DataFrame,
    ul_bursts: pd.DataFrame,
    simulation_duration: float,
    params: Dict,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    DL placed per-flow with independent random start offsets.
    Single synthetic session-open UL burst at t≈0.
    No causal coupling — server pushes whenever it wants.
    """
    jitter     = params.get("jitter", 0.005)
    records_dl = []

    if not dl_bursts.empty:
        for _, flow_df in dl_bursts.groupby("synthetic_flow_id"):
            flow_df  = flow_df.sort_values("synthetic_burst_idx")
            flow_dur = (flow_df["on_dur_s"].sum()
                        + flow_df["off_dur_s"].fillna(0).sum())
            fs = _safe_flow_start(flow_dur, simulation_duration)
            records_dl.extend(_place_bursts_sequentially(flow_df, fs, jitter))

    t_open     = float(np.random.uniform(0.0, 0.5))
    records_ul = [_make_minimal_ul_burst(0, t_open)]

    return (
        _scale_to_window(
            pd.DataFrame(records_dl) if records_dl else pd.DataFrame(),
            simulation_duration,
        ),
        pd.DataFrame(records_ul),
    )

# ══════════════════════════════════════════════════════════════════════════════
#  PART 4 — DISPATCHER
# ══════════════════════════════════════════════════════════════════════════════

_MODE_FUNCTIONS = {
    "streaming"              : align_streaming,
    "request_response"       : align_request_response,
    "bidirectional_continuous": align_bidirectional_continuous,
    "uplink_push"            : align_uplink_push,
    "iot_periodic"           : align_iot_periodic,
    "server_push"            : align_server_push,
    "independent"            : align_independent,
}


# ══════════════════════════════════════════════════════════════════════════════
#  PART 5 — MAIN LOOP
# ══════════════════════════════════════════════════════════════════════════════

ue_burst_data: Dict[str, Dict] = {}

print("=" * 72)
print("  TIME-ALIGNING BURSTS — AUTO MODE DETECTION")
print("=" * 72)

correlation_enabled = TEMPORAL_CORRELATION.get("enabled", True)

if correlation_enabled:
    app_modes = resolve_app_modes(APPS, app_data, TEMPORAL_CORRELATION)
else:
    app_modes = {
        app: ("independent", "correlation disabled") for app in APPS
    }

print("\n  Mode detection report (computed once from full pool):\n")
for app, (mode, reason) in app_modes.items():
    fp = compute_app_fingerprint(app_data[app]["dl"], app_data[app]["ul"])
    print(f"  📱 {app.upper()}")
    print(f"     byte_ratio={fp['byte_ratio']:.1f}  "
          f"ul_byte_share={fp['ul_byte_share']:.3f}  "
          f"dl_cv_gap={fp['dl_cv_gap']:.2f}  "
          f"ul_cv_gap={fp['ul_cv_gap']:.2f}  "
          f"median_dl_on_dur={fp['median_dl_on_dur']:.3f}s  "
          f"median_pkt_size={fp['median_pkt_size']:.0f}B")
    print(f"     → mode  : {mode.upper()}")
    print(f"       reason: {reason}")
    print()

print("  To override: TEMPORAL_CORRELATION['mode_overrides'] = "
      "{'app_name': 'mode'}")
print(f"  Valid modes: {sorted(VALID_MODES)}")
print()

print("=" * 72)
print("  ALIGNING BURSTS PER UE")
print("=" * 72)

for ue_name, assignment in ue_flow_assignments.items():
    print(f"\n  {ue_name} ({assignment['class']}):")
    ue_bursts: Dict[str, Dict] = {}

    for app in APPS:
        fa     = assignment["flows"][app]
        dl_df  = app_data[app]["dl"]
        ul_df  = app_data[app]["ul"]

        dl_raw = extract_bursts(dl_df, fa["dl_flows"])
        ul_raw = extract_bursts(ul_df, fa["ul_flows"])

        mode, _ = app_modes[app]
        fn      = _MODE_FUNCTIONS.get(mode, align_independent)
        dl_aligned, ul_aligned = fn(
            dl_raw, ul_raw, SIMULATION_DURATION, TEMPORAL_CORRELATION
        )

        ue_bursts[app] = {"dl": dl_aligned, "ul": ul_aligned}

        # show true active window (min_start–max_end)
        def _span(df: pd.DataFrame) -> str:
            if df.empty:
                return "—"
            t0 = df["absolute_start_time"].min()
            t1 = df["absolute_end_time"].max()
            return f"{t0:.1f}–{t1:.1f}s"

        dl_mb = dl_aligned["bytes"].sum() / 1e6 if not dl_aligned.empty else 0
        ul_kb = ul_aligned["bytes"].sum() / 1e3 if not ul_aligned.empty else 0

        print(f"    {app} [{mode}]:")
        print(f"      DL : {len(dl_aligned):>5} bursts  "
              f"{dl_mb:>7.2f} MB  span {_span(dl_aligned)}")
        print(f"      UL : {len(ul_aligned):>5} bursts  "
              f"{ul_kb:>7.2f} KB  span {_span(ul_aligned)}")

    ue_burst_data[ue_name] = ue_bursts

print("\n  ✅  Cell 6 complete")

## Cell 7 — Validate & Save

Runs sanity checks then writes everything to a timestamped run directory.
The run name includes the app list so it's identifiable at a glance.

___________________________________________________________________

- fixes the broken timeline-summary logic
- keeps stronger DL thresholds
- treats timeline overflow as an ERROR (not just a warning)
- keeps mode-aware validation
- saves validation results into config.json


In [ ]:
from __future__ import annotations

import json
from datetime import datetime
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd


# ══════════════════════════════════════════════════════════════════════════════
#  THRESHOLDS FOR CHECK 1
#  Tune these intentionally want to allow sparser UEs.
# ══════════════════════════════════════════════════════════════════════════════

MIN_DL_BURSTS_PER_UE = 10      # minimum DL burst count per UE
MIN_DL_MB_PER_UE     = 0.05    # minimum meaningful DL volume per UE (50 KB)
MIN_DL_SPAN_S_PER_UE = 0.1     # minimum active DL window span (seconds)


# ══════════════════════════════════════════════════════════════════════════════
#  VALIDATION HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def _concat_direction(
    bursts_by_app: Dict[str, Dict[str, pd.DataFrame]],
    direction: str,
) -> pd.DataFrame:
    """Concatenate one direction across all apps for a UE."""
    dfs = [
        bursts[direction]
        for bursts in bursts_by_app.values()
        if direction in bursts and not bursts[direction].empty
    ]
    if not dfs:
        return pd.DataFrame()
    return pd.concat(dfs, ignore_index=True)


def _active_window_span(df: pd.DataFrame) -> float:
    """Return max_end - min_start for a dataframe with absolute timestamps."""
    if df.empty:
        return 0.0
    return float(df["absolute_end_time"].max() - df["absolute_start_time"].min())


def _check_timestamps(df: pd.DataFrame, label: str) -> List[str]:
    """Return hard errors for timestamp anomalies."""
    errors: List[str] = []
    if df.empty:
        return errors

    neg = int((df["absolute_start_time"] < 0).sum())
    if neg:
        errors.append(f"{label}: {neg} burst(s) with negative start time")

    inv = int((df["absolute_end_time"] < df["absolute_start_time"]).sum())
    if inv:
        errors.append(f"{label}: {inv} burst(s) where end < start")

    return errors


def _check_flow_monotonicity(
    df: pd.DataFrame,
    label: str,
    mode: str = "other",
) -> List[str]:
    """
    Check for intra-group timestamp overlaps.

    Uses timeline_flow_id when available (written by align_request_response).
    That field gives each scheduled burst-window its own unique identity,
    separate from the original synthetic_flow_id.  This prevents false
    positives when the same synthetic_flow_id is intentionally spread across
    multiple request-response batches at different times.

    Falls back to synthetic_flow_id for all other modes, where the original
    flow identity correctly describes one contiguous scheduled timeline.
    """
    
    errors: List[str] = []
    if df.empty or "synthetic_flow_id" not in df.columns:
        return errors

    # Prefer timeline_flow_id (set by align_request_response) when present
    flow_col = (
        "timeline_flow_id"
        if "timeline_flow_id" in df.columns
        else "synthetic_flow_id"
    )

    for flow_id, g in df.groupby(flow_col):
        g = g.sort_values("absolute_start_time")
        starts = g["absolute_start_time"].to_numpy()
        ends   = g["absolute_end_time"].to_numpy()

        if len(starts) <= 1:
            continue

        overlap = (starts[1:] < ends[:-1]).sum()
        if overlap:
            errors.append(
                f"{label}: flow {flow_id} has {int(overlap)} intra-flow overlap(s)"
            )

    return errors


def _validate_mode(
    ue_name: str,
    app: str,
    mode: str,
    dl: pd.DataFrame,
    ul: pd.DataFrame,
    simulation_duration: float,
    params: Dict,
) -> Tuple[List[str], List[str]]:
    """
    Mode-specific validation.
    Returns (warnings, errors).
    """
    warnings_out: List[str] = []
    errors_out: List[str] = []
    prefix = f"{ue_name}/{app}"

    # ── request_response ─────────────────────────────────────────────────────
    
    if mode == "request_response":
        if ul.empty:
            warnings_out.append(
                f"{prefix} [request_response]: no UL bursts — request triggers are missing"
            )
        elif not dl.empty:
            rtt_min, rtt_max = params.get("rtt_delay_range", (0.010, 0.050))
            jitter           = params.get("jitter", 0.005)

            # allow some slack because alignment may include small jitter
            
            gap_lo = rtt_min - jitter
            gap_hi = rtt_max + jitter + 2.0   # +2 s slack for idle gap / wrap

            has_batch_id = (
                "correlated_batch_id" in dl.columns
                and "correlated_batch_id" in ul.columns
            )

            if has_batch_id:
                batch_ids = np.array(sorted(dl["correlated_batch_id"].unique()))
                if len(batch_ids) > 20:
                    batch_ids = np.random.choice(batch_ids, size=20, replace=False)

                early = late = missing = 0

                for bid in batch_ids:
                    ul_batch = ul[ul["correlated_batch_id"] == bid]
                    dl_batch = dl[dl["correlated_batch_id"] == bid]

                    if ul_batch.empty or dl_batch.empty:
                        missing += 1
                        continue

                    ul_end   = float(ul_batch["absolute_end_time"].max())
                    dl_start = float(dl_batch["absolute_start_time"].min())
                    gap      = dl_start - ul_end

                    if gap < gap_lo:
                        early += 1
                    if gap > gap_hi:
                        late += 1

                n = len(batch_ids)
                if early:
                    warnings_out.append(
                        f"{prefix} [request_response]: {early}/{n} sampled batch(es) "
                        f"have DL starting before expected RTT "
                        f"(gap < {rtt_min:.3f}s)"
                    )
                if late:
                    warnings_out.append(
                        f"{prefix} [request_response]: {late}/{n} sampled batch(es) "
                        f"have DL starting > rtt_max + 2s after UL end"
                    )
                if missing:
                    warnings_out.append(
                        f"{prefix} [request_response]: {missing}/{n} sampled batch(es) "
                        f"had no matching UL or DL rows"
                    )
            else:
                dl_first = float(dl["absolute_start_time"].min())
                ul_first = float(ul["absolute_start_time"].min())
                if ul_first > dl_first + 0.1:
                    warnings_out.append(
                        f"{prefix} [request_response]: first UL start "
                        f"({ul_first:.3f}s) is after first DL start "
                        f"({dl_first:.3f}s) by > 100 ms "
                        f"(coarse fallback check only)"
                    )

    # ── streaming ─────────────────────────────────────────────────────────────
    
    elif mode == "streaming":
        if dl.empty:
            errors_out.append(f"{prefix} [streaming]: no DL bursts")
        else:
            dl_span = _active_window_span(dl)
            if dl_span < simulation_duration * 0.10:
                warnings_out.append(
                    f"{prefix} [streaming]: DL active span={dl_span:.1f}s "
                    f"covers < 10% of simulation window"
                )

        if not dl.empty and not ul.empty:
            ul_bytes = float(ul["bytes"].sum())
            dl_bytes = float(dl["bytes"].sum())
            ratio    = dl_bytes / ul_bytes if ul_bytes > 0 else float("inf")
            if ratio < 3.0:
                warnings_out.append(
                    f"{prefix} [streaming]: DL/UL byte ratio={ratio:.1f} "
                    f"lower than expected (< 3)"
                )

    # ── server_push ───────────────────────────────────────────────────────────
    
    elif mode == "server_push":
        if dl.empty:
            errors_out.append(f"{prefix} [server_push]: no DL bursts")

        if not ul.empty and len(ul) > 3:
            warnings_out.append(
                f"{prefix} [server_push]: {len(ul)} UL bursts — expected very few"
            )

    # ── uplink_push ───────────────────────────────────────────────────────────
    
    elif mode == "uplink_push":
        if ul.empty:
            errors_out.append(f"{prefix} [uplink_push]: no UL bursts")
        elif not dl.empty:
            ul_b = float(ul["bytes"].sum())
            dl_b = float(dl["bytes"].sum())
            ratio = ul_b / dl_b if dl_b > 0 else float("inf")
            if ratio < 2.0:
                warnings_out.append(
                    f"{prefix} [uplink_push]: UL/DL byte ratio={ratio:.1f} "
                    f"lower than expected for upload-heavy traffic (< 2)"
                )

    # ── bidirectional_continuous ─────────────────────────────────────────────
    
    elif mode == "bidirectional_continuous":
        for direction, df_dir in [("DL", dl), ("UL", ul)]:
            if df_dir.empty:
                errors_out.append(
                    f"{prefix} [bidirectional_continuous]: no {direction} bursts"
                )
            else:
                span = _active_window_span(df_dir)
                if span < simulation_duration * 0.05:
                    warnings_out.append(
                        f"{prefix} [bidirectional_continuous]: {direction} span={span:.1f}s "
                        f"covers < 5% of simulation window"
                    )

    # ── iot_periodic ─────────────────────────────────────────────────────────
    
    elif mode == "iot_periodic":
        for direction, df_dir in [("DL", dl), ("UL", ul)]:
            if df_dir.empty:
                continue
            gaps = df_dir["off_dur_s"].dropna() if "off_dur_s" in df_dir.columns else pd.Series([], dtype=float)
            if len(gaps) > 1 and gaps.mean() > 0:
                cv = float(gaps.std() / gaps.mean())
                if cv > 0.5:
                    warnings_out.append(
                        f"{prefix} [iot_periodic]: {direction} gap CV after alignment = {cv:.2f} "
                        f"— regularity may have been distorted"
                    )

    # ── independent ──────────────────────────────────────────────────────────
    
    elif mode == "independent":
        if dl.empty:
            warnings_out.append(f"{prefix} [independent]: no DL bursts after alignment")

    return warnings_out, errors_out


# ══════════════════════════════════════════════════════════════════════════════
#  MAIN VALIDATION
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 72)
print("  VALIDATION")
print("=" * 72)

all_warnings: List[str] = []
all_errors: List[str] = []

# ── Check 1: minimum DL traffic per UE ───────────────────────────────────────

print("\n  Check 1 — minimum DL traffic per UE")
print(f"    bursts ≥ {MIN_DL_BURSTS_PER_UE}  |  "
      f"MB ≥ {MIN_DL_MB_PER_UE}  |  "
      f"span ≥ {MIN_DL_SPAN_S_PER_UE}s")
print("    (if UEs fail the MB threshold, reduce N_UE or increase flow pool size)\n")

for ue_name, bursts_by_app in ue_burst_data.items():
    dl_all = _concat_direction(bursts_by_app, "dl")

    total_dl_bursts = int(len(dl_all))
    total_dl_mb     = float(dl_all["bytes"].sum() / 1e6) if not dl_all.empty else 0.0
    dl_span         = _active_window_span(dl_all)

    ok_bursts = total_dl_bursts >= MIN_DL_BURSTS_PER_UE
    ok_mb     = total_dl_mb     >= MIN_DL_MB_PER_UE
    ok_span   = dl_span         >= MIN_DL_SPAN_S_PER_UE
    ok        = ok_bursts and ok_mb and ok_span

    failed = []
    if not ok_bursts:
        failed.append(f"bursts={total_dl_bursts}<{MIN_DL_BURSTS_PER_UE}")
    if not ok_mb:
        failed.append(f"MB={total_dl_mb:.3f}<{MIN_DL_MB_PER_UE}")
    if not ok_span:
        failed.append(f"span={dl_span:.1f}s<{MIN_DL_SPAN_S_PER_UE}s")

    print(
        f"    {'✅' if ok else '❌'} {ue_name} : "
        f"{total_dl_bursts} bursts  "
        f"{total_dl_mb:.3f} MB  "
        f"span {dl_span:.1f}s"
        + (f"  ← {', '.join(failed)}" if failed else "")
    )

    if not ok:
        all_errors.append(
            f"{ue_name}: DL traffic below minimum threshold "
            f"({', '.join(failed)})"
        )

# ── Check 2: timeline sanity ──────────────────────────────────────────────────

print(f"\n  Check 2 — timeline sanity "
      f"(≤ {SIMULATION_DURATION}s, no negatives, no inversions, no intra-flow overlap)")

timeline_clean = True

for ue_name, bursts_by_app in ue_burst_data.items():
    for app, bursts in bursts_by_app.items():
        app_mode, _ = app_modes[app]   # needed for mode-aware overlap check
        for direction, df in [("DL", bursts["dl"]), ("UL", bursts["ul"])]:
            if df.empty:
                continue

            label = f"{ue_name}/{app} {direction}"
            max_t = float(df["absolute_end_time"].max())

            if max_t > SIMULATION_DURATION:
                msg = f"{label}: ends {max_t:.1f}s > {SIMULATION_DURATION}s"
                print(f"    ❌ {msg}")
                all_errors.append(msg)
                timeline_clean = False

            for e in _check_timestamps(df, label):
                print(f"    ❌ {e}")
                all_errors.append(e)
                timeline_clean = False

            for e in _check_flow_monotonicity(df, label, app_mode):
                print(f"    ❌ {e}")
                all_errors.append(e)
                timeline_clean = False

if timeline_clean:
    print(f"    ✅  All timelines valid and within {SIMULATION_DURATION}s")

# ── Check 3: mode-aware validation ───────────────────────────────────────────

print("\n  Check 3 — mode-aware validation")

check3_clean = True

for ue_name, bursts_by_app in ue_burst_data.items():
    for app, bursts in bursts_by_app.items():
        mode, _ = app_modes[app]
        warns, errs = _validate_mode(
            ue_name=ue_name,
            app=app,
            mode=mode,
            dl=bursts["dl"],
            ul=bursts["ul"],
            simulation_duration=SIMULATION_DURATION,
            params=TEMPORAL_CORRELATION,
        )

        for w in warns:
            print(f"    ⚠️  {w}")
            all_warnings.append(w)
            check3_clean = False

        for e in errs:
            print(f"    ❌ {e}")
            all_errors.append(e)
            check3_clean = False

if check3_clean:
    print("    ✅  All mode-specific checks passed")

# ── Summary ───────────────────────────────────────────────────────────────────

print()

if all_errors:
    print(f"  ❌  {len(all_errors)} error(s) — fix before deploying:")
    for e in all_errors:
        print(f"       • {e}")
    print()
    print("  Common fixes:")
    print("    • UE starved (low MB / low bursts): reduce N_UE or increase flow pool size.")
    print("    • Wrong mode: add to TEMPORAL_CORRELATION['mode_overrides'].")
if all_warnings:
    print(f"  ⚠️   {len(all_warnings)} warning(s) — review if unexpected:")
    for w in all_warnings:
        print(f"       • {w}")
if not all_errors and not all_warnings:
    print("  ✅  All checks passed.")
elif not all_errors:
    print("  ✅  No errors (warnings above are informational).")


# ══════════════════════════════════════════════════════════════════════════════
#  SAVE
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 72)
print("  SAVING")
print("=" * 72)

timestamp  = datetime.now().strftime("%Y%m%d_%H%M%S")
app_str    = "_".join(APPS)
run_name   = f"run_{app_str}_{timestamp}"
run_dir    = OUTPUT_ROOT / run_name
run_dir.mkdir(parents=True, exist_ok=True)

bursts_dir = run_dir / "bursts"
bursts_dir.mkdir(exist_ok=True)

print(f"\n  Run directory: {run_dir.relative_to(PROJECT_ROOT)}")

config = {
    "run_name": run_name,
    "timestamp": timestamp,
    "apps": APPS,
    "simulation_duration": SIMULATION_DURATION,
    "n_ue": N_UE,
    "user_class_mode": USER_CLASS_MODE,
    "user_class_distribution": USER_CLASS_DISTRIBUTION,
    "sampling_strategy": SAMPLING_STRATEGY,
    "temporal_correlation": TEMPORAL_CORRELATION,
    "app_traffic_modes": {
        app: mode for app, (mode, _) in app_modes.items()
    },
    "network": {
        "dn_ip": DN_IP,
        "ue_ip_prefix": UE_IP_PREFIX,
        "ue_ip_start": UE_IP_START,
        "dl_port": DL_PORT,
        "ul_port": UL_PORT,
    },
    "random_seed": RANDOM_SEED,
    "validation": {
        "errors": all_errors,
        "warnings": all_warnings,
        "passed": not bool(all_errors),
        "thresholds": {
            "MIN_DL_BURSTS_PER_UE": MIN_DL_BURSTS_PER_UE,
            "MIN_DL_MB_PER_UE": MIN_DL_MB_PER_UE,
            "MIN_DL_SPAN_S_PER_UE": MIN_DL_SPAN_S_PER_UE,
        },
    },
}
(run_dir / "config.json").write_text(json.dumps(config, indent=2))
print("  ✅  config.json  (includes app_traffic_modes and validation log)")

rows = []
for ue_name, asgn in ue_flow_assignments.items():
    row = {"ue_name": ue_name, "ue_class": asgn["class"]}
    for app in APPS:
        row[f"{app}_dl_flows"] = str(asgn["flows"][app]["dl_flows"])
        row[f"{app}_ul_flows"] = str(asgn["flows"][app]["ul_flows"])
    rows.append(row)

pd.DataFrame(rows).to_csv(run_dir / "ue_flow_assignments.csv", index=False)
print("  ✅  ue_flow_assignments.csv")

for ue_name, bursts_by_app in ue_burst_data.items():
    for app, bursts in bursts_by_app.items():
        for direction, df in [("dl", bursts["dl"]), ("ul", bursts["ul"])]:
            if df.empty:
                continue
            fname = bursts_dir / f"{ue_name}_{app}_{direction}_bursts.parquet"
            df.to_parquet(fname, index=False)
            print(f"  ✅  bursts/{fname.name}")

print(f"\n  Run saved → {run_dir.relative_to(PROJECT_ROOT)}")
if all_errors:
    print("\n  ⚠️  Saved despite validation errors — review before deploying.")
print("\n  ✅  Cell 7 complete — ready (MGEN script generation)")